# Demo en vivo · llegada de registros al pipeline

Simula eventos que llegan **uno por uno** y muestra, etapa por etapa, qué decide el pipeline.
Úsalo en la exposición: cada escenario es una celda; ejecútala y explica lo que aparece.

| Escenario | Qué se ve | Etapas |
|---|---|---|
| 1 | Una compra express real atraviesa todo el pipeline y recibe recomendaciones | 0 → 1 → 2 → 6 → 3 → 4 → 9 |
| 2 | El mismo registro llega dos veces: el segundo se descarta | 1 |
| 3 | Un registro con precio negativo se descarta | 1 |
| 4 | Seis eventos mezclados: el heap atiende primero lo urgente | 2 |
| 5 | Un usuario de la lista negra: Bloom Filter + verificación exacta | 6 |
| 6 | Un bot agrega 20 carritos en 2 minutos: Count-Min Sketch dispara la alerta | 6 |
| 7 | Tres pagos rechazados con la misma tarjeta: prueba de tarjetas | 6 |
| 8 | Autocompletado, rango de precios, más vendidos | 9 |
| 9 | Llegan 20,000 eventos más: micro-lote y la recomendación cambia | 5, 7, 8 |
| 10 | Similitud aproximada frente al método exacto (LSH vs Jaccard, HNSW vs KNN) | 7 |
| 11 | Análisis del grafo: PageRank vs grado, Louvain vs Label Propagation | 8 |

**Cómo cubre el criterio de Implementación (4 pts):**

| Lo que exige el enunciado | Dónde se ve en la demo |
|---|---|
| Un evento atraviesa las diez etapas y produce una salida observable | Escenario 1 (etapas 0→1→2→6→3→4→5/7/8→9 y recomendación) |
| Prioridades | Escenario 4 (y nivel del heap en cada evento) |
| Filtrado probabilístico | Escenarios 5, 6 y 7 (Bloom Filter y Count-Min Sketch con verificación exacta) |
| Similitud frente a un método exacto | Escenario 10 |
| Análisis del grafo | Escenario 11 (y puesto en PageRank en el escenario 1) |
| Servicio al usuario final | Escenarios 1, 8 y 9 (recomendaciones, autocompletado, Top-K, rango) |

**Tiempo:** preparar los datos y el calentamiento toma unos minutos; luego cada escenario es instantáneo.
Ejecuta las celdas 1 y 2 **antes** de la exposición.

## 1 · Preparar (ejecutar antes de exponer)

In [14]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE = '/content/drive/MyDrive/Proyecto EDA'
CSV = f'{DRIVE}/2019-Oct.csv.zip'

if not os.path.exists('/content/proyecto/src'):
    !git clone -q https://github.com/AMPeru12/EADA.git /content/proyecto
%cd /content/proyecto
!git pull -q origin main
!pip install -q -r requirements.txt

# Datos: si ya existe un stream de pasos anteriores se reutiliza; si no, se generan 300,000 eventos (1–2 min)
if not glob.glob('data/stream_*'):
    !python src/preparar_datos.py --input "$CSV" --n 300000 --out data
STREAM = sorted(glob.glob('data/stream_*'))[0]
print('Stream:', STREAM)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/proyecto
Stream: data/stream_300000.parquet


In [15]:
from src.demo import DemoEnVivo
# Calentamiento: procesa 200,000 eventos reales para que existan catálogo, índices y grafo.
# pausa = segundos entre etapas (súbelo a 0.8 si quieres que se lea con calma en la exposición)
d = DemoEnVivo(STREAM, 'data/blacklist.csv', 'data/catalogo_stock.csv', n_calentamiento=200_000, pausa=0.6)

Cargando 200,000 eventos reales para el calentamiento...
Listo en 62 s: 29,382 productos, 3,652 alertas, índices y grafo construidos.



## 2 · Escenarios
**Escenario 1 · Compra express.** Qué decir: *"Este evento es real. Entra a la fila, pasa el control de calidad,
el heap lo atiende con prioridad 1 por ser express, el guardia de fraude lo aprueba, se descuenta el stock,
se indexa por categoría y precio, y el servicio devuelve recomendaciones de cuatro estructuras distintas."*

In [16]:
d.compra_normal()


▶ Escenario 1 · Una compra express real
  [0 Ingesta · Cola FIFO ] llega compra del usuario 537463105 (producto 5100816, precio 29.51) y entra a la fila
  [1 Preprocesamiento    ] ✅ válido, normalizado y sin duplicado
  [2 Prioridad · Heap    ] nivel 1 (express)
  [6 Fraude · Bloom      ] Bloom: usuario, tarjeta y dispositivo NO están en la lista negra (seguro)
  [6 Fraude · Count-Min  ] compras del usuario en 10 min ≈ 1 (umbral 6) → normal
  [3 Almacén · Hash/LRU  ] ✅ ficha del producto 5100816 en la tabla hash · stock 64 → 63 · orden registrada
  [4 Índice · B+Tree     ] indexado por (categoría, precio 29.51); 4 productos de su categoría con ±10% de precio y stock
  [5 KD-Tree             ] producto en el índice de atributos: sí (28,948 productos)
  [7 MinHash+LSH / HNSW  ] firma de 256 valores en LSH: sí · embedding en HNSW: sí
  [8 Grafo · PageRank    ] nodo del grafo con 3 productos co-comprados · puesto 116 de 2,446 en PageRank
  [9 Servicio            ] recomendaciones para est

'aprobado'

**Escenario 2 · Duplicado.** *"Un doble clic manda el mismo registro dos veces. La segunda vez se detecta con ordenación y búsqueda binaria y se descarta."*

In [17]:
d.duplicado()


▶ Escenario 2 · El mismo registro llega dos veces (por ejemplo, un doble clic)

▶ Primera vez:
  [0 Ingesta · Cola FIFO ] llega carrito del usuario 550324822 (producto 1004767, precio 254.76) y entra a la fila
  [1 Preprocesamiento    ] ✅ válido, normalizado y sin duplicado
  [2 Prioridad · Heap    ] nivel 4 (carrito/búsqueda)
  [6 Fraude · Bloom      ] Bloom: usuario, tarjeta y dispositivo NO están en la lista negra (seguro)
  [6 Fraude · Count-Min  ] carritos del usuario en 10 min ≈ 5 (umbral 18) → normal
  [3 Almacén · Hash/LRU  ] ✅ ficha del producto 1004767 en la tabla hash
  [4 Índice · B+Tree     ] indexado por (categoría, precio 254.76); 64 productos de su categoría con ±10% de precio y stock

▶ Segunda vez (idéntico):
  [0 Ingesta · Cola FIFO ] llega carrito del usuario 550324822 (producto 1004767, precio 254.76) y entra a la fila
  [1 Preprocesamiento    ] ❌ DESCARTADO: es un duplicado del registro anterior


**Escenario 3 · Registro inválido.** *"Un precio negativo no tiene sentido: se descarta y se guarda el motivo."*

In [18]:
d.invalido()


▶ Escenario 3 · Registro con precio negativo
  [0 Ingesta · Cola FIFO ] llega vista del usuario 516041820 (producto 11100270, precio -5.0) y entra a la fila
  [1 Preprocesamiento    ] ❌ DESCARTADO por inválido (precio_invalido)


**Escenario 4 · Prioridades.** *"Llegan seis eventos a la vez. El heap no los atiende por orden de llegada sino por urgencia, como una sala de emergencias."*

In [19]:
d.prioridades()


▶ Escenario 4 · Llegan 6 eventos mezclados al mismo tiempo; el heap decide el orden
  Orden de llegada:   vista → programada → carrito → express → estándar → vista
  Orden de atención:  express → estándar → programada → carrito → vista → vista
  (el heap siempre entrega primero el más urgente, sin reordenar toda la fila)


**Escenario 5 · Lista negra.** *"El Bloom Filter dice 'quizás está en la lista'; antes de bloquear se confirma con la lista exacta."*

In [20]:
d.lista_negra()


▶ Escenario 5 · Compra de un usuario que está en la lista negra (546343539)
  [0 Ingesta · Cola FIFO ] llega compra del usuario 546343539 (producto 1002629, precio 377.14) y entra a la fila
  [1 Preprocesamiento    ] ✅ válido, normalizado y sin duplicado
  [2 Prioridad · Heap    ] nivel 2 (estándar)
  [6 Fraude · Bloom      ] Bloom dice 'quizás' para ['user:546343539'] → lista exacta: CONFIRMADO
  [6 Fraude              ] 🚨 ALERTA 'lista_negra' → vuelve al heap con prioridad 0; la operación se BLOQUEA


**Escenario 6 · Bot.** *"Un usuario que no está en ninguna lista agrega 20 productos en 2 minutos. El Count-Min Sketch lleva la cuenta con memoria mínima; al llegar a 18 se verifica con el conteo exacto y se alerta."*

In [21]:
d.rafaga_bot()


▶ Escenario 6 · Un bot agrega 20 productos al carrito en 2 minutos (usuario nuevo 999000001, no está en la lista negra)
  carrito  1: Count-Min ≈  1 → aprobado
  carrito  2: Count-Min ≈  2 → aprobado
  carrito  3: Count-Min ≈  3 → aprobado
  carrito  4: Count-Min ≈  4 → aprobado
  carrito  5: Count-Min ≈  5 → aprobado
  carrito  6: Count-Min ≈  6 → aprobado
  carrito  7: Count-Min ≈  7 → aprobado
  carrito  8: Count-Min ≈  8 → aprobado
  carrito  9: Count-Min ≈  9 → aprobado
  carrito 10: Count-Min ≈ 10 → aprobado
  carrito 11: Count-Min ≈ 11 → aprobado
  carrito 12: Count-Min ≈ 12 → aprobado
  carrito 13: Count-Min ≈ 13 → aprobado
  carrito 14: Count-Min ≈ 14 → aprobado
  carrito 15: Count-Min ≈ 15 → aprobado
  carrito 16: Count-Min ≈ 16 → aprobado
  carrito 17: Count-Min ≈ 17 → aprobado
  [6 Fraude              ] 🚨 ALERTA 'intensidad' (Count-Min estimó 18, verificado con el conteo exacto) → vuelve al heap con prioridad 0; la operación se BLOQUEA
  carrito 18: Count-Min ≈ 18 → alerta

**Escenario 7 · Prueba de tarjetas.** *"Cada pago rechazado genera una alerta de pago; el tercer rechazo de la misma tarjeta en 10 minutos indica que alguien está probando tarjetas robadas."*

In [22]:
d.prueba_tarjetas()


▶ Escenario 7 · Tres pagos rechazados con la misma tarjeta en pocos minutos

▶ Intento 1:
  [0 Ingesta · Cola FIFO ] llega compra del usuario 516606449 (producto 1002524, precio 514.76) y entra a la fila
  [1 Preprocesamiento    ] ✅ válido, normalizado y sin duplicado
  [2 Prioridad · Heap    ] nivel 2 (estándar)
  [6 Fraude · Bloom      ] Bloom: usuario, tarjeta y dispositivo NO están en la lista negra (seguro)
  [6 Fraude              ] 🚨 ALERTA 'pago_rechazado' → vuelve al heap con prioridad 0; alerta de pago para el equipo de riesgo (el pago ya fue rechazado)

▶ Intento 2:
  [0 Ingesta · Cola FIFO ] llega compra del usuario 516606449 (producto 1002524, precio 514.76) y entra a la fila
  [1 Preprocesamiento    ] ✅ válido, normalizado y sin duplicado
  [2 Prioridad · Heap    ] nivel 2 (estándar)
  [6 Fraude · Bloom      ] Bloom: usuario, tarjeta y dispositivo NO están en la lista negra (seguro)
  [6 Fraude              ] 🚨 ALERTA 'pago_rechazado' → vuelve al heap con prioridad 0; al

**Escenario 8 · Servicio.** *"Lo que ve el cliente: autocompletado con el Trie, los más vendidos con Top-K y la consulta por rango de precio con el B+Tree."*

In [23]:
d.servicio('sa')   # cambia el prefijo: 'ap', 'xi', 'le'...


▶ Escenario 8 · Lo que ve el cliente y el gerente (etapa 9)
  [Trie                  ] autocompletar 'sa': ['samsung', 'saturn', 'salamander', 'sailun', 'sava']
  [Top-K                 ] 10 más vendidos: [1004856, 1004767, 1004870, 1002544, 1004833, 4804056, 1005115, 1004873, 1004741, 5100816]
  [B+Tree                ] categoría 2053013565983425517, precio 50–500, con stock: 222 productos, por ejemplo [3701387, 3701404, 3700605, 3701379, 3701187]
  [Alertas               ] total 3,657


**Escenario 9 · Micro-lote.** *"Llegan 20,000 eventos reales más. Las etapas caras (KD-Tree, LSH, HNSW y grafo) no se recalculan con cada clic sino por micro-lotes; al refrescarse, la recomendación del producto cambia: el sistema aprende mientras llega el stream."*

In [24]:
d.micro_lote()


▶ Escenario 9 · Llegan 20,000 eventos reales más y se ejecuta un micro-lote (etapas 5, 7 y 8)
  [0–4, 6                ] 20,000 eventos procesados por evento en 1.3 s
  [5/7/8 micro-lote      ] en 14.59 s: 30,895 productos · 5,417 firmas LSH (3,230 actualizadas) · 5,000 vectores HNSW · 17,434 aristas
      · Se compra junto con del producto 5100816: [5300946, 1003306, 1002524, 4804295, 1005115] → [5100826, 5100818, 5100817, 5100820, 5100821] (cambió)
      · Similares del producto 5100816: [5100833] → [5100833, 5100551, 5100375] (cambió)


**Escenario 10 · Similitud frente al método exacto.** *"LSH responde en milisegundos lo que la fuerza bruta calcula comparando contra todos los productos, y encuentra los pares con Jaccard ≥ 0.125. HNSW recupera los mismos 10 vecinos que el KNN exacto."*

In [25]:
d.similitud_vs_exacto()


▶ Escenario 10 · Similitud aproximada frente al método exacto (producto 1005115)
  [MinHash+LSH           ] 3 candidatos en 0.4 ms: [1005105, 1005135, 1005116]
  [Jaccard exacto        ] compara contra 5,417 productos en 526.7 ms; los 5 más similares: 1005105 (J=0.17), 1004249 (J=0.09), 1003317 (J=0.06), 1005135 (J=0.06), 1005104 (J=0.05)
  [Resultado             ] LSH encontró 1 de los 1 productos con Jaccard ≥ 0.125 (recall 1.00) · LSH respondió 1242 veces más rápido
  [HNSW vs KNN exacto    ] 100 consultas de 10 vecinos sobre 5,000 productos: HNSW 96 ms · exacto 54 ms · recall@10 = 0.992
      (con pocos miles de productos la fuerza bruta de NumPy puede ganar; la ventaja de HNSW aparece a gran escala:
       en el benchmark fue 16 veces más rápido que el KNN exacto con 100,000 vectores)


**Escenario 11 · Grafo.** *"PageRank mide influencia y se compara con el grado; Louvain y Label Propagation encuentran grupos de productos que se compran juntos; Personalized PageRank da el cross-selling."*

In [13]:
d.grafo()


▶ Escenario 11 · Análisis del grafo de co-compra (2,603 productos, 17,434 aristas; fuente: compras + carritos (peso 0.5))
  [PageRank (influencia) ] Top-5: [1004767, 1004833, 1004856, 1002544, 1005115]
  [Grado (conexiones)    ] Top-5: [1004767, 16200283, 1004833, 1004856, 13800437] · Spearman con PageRank = 0.179
  [Louvain               ] modularidad 0.972 · 230 comunidades · tamaño máx. 89 · 0.25 s · estabilidad (NMI, 5 semillas) 0.995
  [Label Propagation     ] modularidad 0.928 · 316 comunidades · tamaño máx. 32 · 0.17 s · estabilidad (NMI, 5 semillas) 0.991
  [Comunidad más grande  ] 89 productos que se compran juntos, por ejemplo [1004720, 1004777, 1004805, 1004838, 1004865, 1004886, 1004903, 1004939]
  [Personalized PageRank ] cross-selling para el producto más influyente 1004767: [1004870, 1002544, 1004833, 1004856, 1004750]


## 3 · Tu propio evento (opcional)
Puedes inventar un registro y enviarlo. Cambia los valores y ejecuta.

In [26]:
ev = d._plantilla('purchase')          # toma un evento real como base
ev.update({'order_type': 'express', 'price': 499.90, 'ts': d._nuevo_ts()})
d.procesar(ev, 'Mi evento')


▶ Mi evento
  [0 Ingesta · Cola FIFO ] llega compra del usuario 513316282 (producto 1004874, precio 499.9) y entra a la fila
  [1 Preprocesamiento    ] ✅ válido, normalizado y sin duplicado · llegó tarde, se procesa igual (tolerancia)
  [2 Prioridad · Heap    ] nivel 1 (express)
  [6 Fraude · Bloom      ] Bloom: usuario, tarjeta y dispositivo NO están en la lista negra (seguro)
  [6 Fraude · Count-Min  ] compras del usuario en 10 min ≈ 2 (umbral 6) → normal
  [3 Almacén · Hash/LRU  ] ✅ ficha del producto 1004874 en la tabla hash · stock 88 → 87 · orden registrada
  [4 Índice · B+Tree     ] indexado por (categoría, precio 499.9); 39 productos de su categoría con ±10% de precio y stock


'aprobado'